# Dynamic BIST Rotation with Markowitz Optimization
## Step-by-Step Walkthrough

This notebook walks through the complete pipeline:
1. Data loading & universe overview
2. Macro panel inspection & lag verification
3. Signal generation & composite scores
4. Optimization demo for a single date
5. Full backtest with metrics
6. Visualizations (cumulative returns, drawdown, weights, rolling Sharpe, monthly heatmap)
7. Sector attribution analysis
8. XU100 benchmark comparison
9. Transaction cost impact
10. Sensitivity analysis

In [ ]:
import sys
from pathlib import Path

# Add src to path so all modules are importable
SRC = Path("src").resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

from config import SECTORS, ALL_TICKERS, DATA_DIR, REPORTS_DIR
print(f"Universe: {len(ALL_TICKERS)} tickers across {len(SECTORS)} sectors")

## 1. Data Loading & Universe Overview

In [ ]:
from asset_fetch import load_prices, summary

prices = load_prices()
print(f"Price matrix: {prices.shape[0]} trading days × {prices.shape[1]} assets")
print(f"Date range: {prices.index[0].date()} → {prices.index[-1].date()}")
print()

# Sector grouping
for sector, tickers in SECTORS.items():
    print(f"  {sector}: {', '.join(t.replace('.IS','') for t in tickers)}")

In [ ]:
summary(prices)

In [ ]:
# Normalized price chart (base = 1)
normed = prices / prices.iloc[0]
fig, ax = plt.subplots(figsize=(13, 6))
normed.plot(ax=ax, linewidth=1.2)
ax.set_title('Normalized Prices (Base = 1)', fontsize=14)
ax.set_ylabel('Growth')
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Macro Panel Inspection & Lag Verification

In [ ]:
from macro_fetch import get_macro_panel

macro = get_macro_panel(prices.index)
print(f"Macro panel: {macro.shape[0]} rows × {macro.shape[1]} columns")
print(f"Columns: {macro.columns.tolist()}")
macro.describe().round(2)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

axes[0].plot(macro.index, macro['USDTRY'], color='#e15759', linewidth=1.5)
axes[0].set_title('USD/TRY Exchange Rate (lagged +1 day)')
axes[0].set_ylabel('TRY per USD')

axes[1].plot(macro.index, macro['WACF_RATE'], color='#4e79a7', linewidth=1.5)
axes[1].set_title('WACF Interest Rate (lagged +1 day)')
axes[1].set_ylabel('Rate (%)')

axes[2].plot(macro.index, macro['CPI_YOY'], color='#f28e2b', linewidth=1.5)
axes[2].set_title('CPI YoY Inflation (lagged +1 month)')
axes[2].set_ylabel('YoY (%)')

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ Lag verification:")
print(f"  USD/TRY & WACF: shifted +1 trading day (config.USDTRY_RELEASE_LAG_DAYS = 1)")
print(f"  CPI: shifted +1 month (config.CPI_RELEASE_LAG_MONTHS = 1)")

## 3. Signal Generation & Composite Scores

In [ ]:
from signal_generation import compute_composite_scores, compute_technical_scores, compute_macro_scores

scores = compute_composite_scores(prices, macro)
print(f"Composite scores: {scores.shape[0]} rows × {scores.shape[1]} assets")
print(f"Score range: [{scores.min().min():.3f}, {scores.max().max():.3f}]")
print()
scores.describe().round(3)

In [ ]:
# Score heatmap for the latest date
latest = scores.tail(1).T
latest.columns = ['Score']
latest = latest.sort_values('Score', ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ca02c' if v > 0 else '#d62728' for v in latest['Score']]
latest['Score'].plot.barh(ax=ax, color=colors, edgecolor='white')
ax.set_title(f"Composite Scores as of {scores.index[-1].date()}", fontsize=13)
ax.set_xlabel('Score [-1, +1]')
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 4. Optimization Demo (Single Date)

In [ ]:
from optimization import compute_weights, equal_weight, validate_weight_vector

demo_date = scores.index[-1]
w_opt = compute_weights(prices, scores, demo_date)
w_eq = equal_weight(len(prices.columns))

comparison = pd.DataFrame({
    'Optimized': w_opt.round(4),
    'Equal (1/N)': w_eq.round(4),
    'Score': scores.loc[demo_date].round(3),
})
print(f"Weights as of {demo_date.date()}:\n")
display(comparison)

val = validate_weight_vector(w_opt)
print(f"\nValidation: sum={val['weight_sum']:.6f}, "
      f"min={val['min_weight']:.6f}, max={val['max_weight']:.6f}")
print(f"All checks pass: {val['sum_to_one'] and val['non_negative'] and val['within_cap']}")

## 5. Full Backtest

In [ ]:
from backtest import run_backtest, performance_metrics, validate_backtest_results

results = run_backtest(prices, scores)
strat_ret = results['strategy_returns']
bench_ret = results['benchmark_returns']
weights_df = results['weights_history']
rebal_dates = results['rebalance_dates']

strat_metrics = performance_metrics(strat_ret, name='Strategy')
bench_metrics = performance_metrics(bench_ret, name='Benchmark (1/N)')
report = pd.DataFrame([strat_metrics, bench_metrics])
display(report)

print("\nValidation checks:")
for name, passed in validate_backtest_results(results, scores).items():
    print(f"  {'✅' if passed else '❌'} {name}")

## 6. Core Visualizations

In [ ]:
from backtest import plot_cumulative_returns, plot_drawdown, plot_weights_over_time

plot_cumulative_returns(strat_ret, bench_ret)
plt.show()

plot_drawdown(strat_ret, bench_ret)
plt.show()

plot_weights_over_time(weights_df)
plt.show()

In [ ]:
from advanced_analytics import plot_rolling_sharpe, plot_monthly_heatmap

plot_rolling_sharpe(strat_ret, bench_ret)
plt.show()

plot_monthly_heatmap(strat_ret, bench_ret)
plt.show()

## 7. Sector Attribution Analysis

In [ ]:
from advanced_analytics import compute_sector_attribution, plot_sector_attribution

attr = compute_sector_attribution(prices, weights_df, rebal_dates)

# Cumulative contribution by sector
print("Cumulative sector contribution (% of total return):")
cum_attr = (attr.sum() * 100).round(2)
display(cum_attr.to_frame('Contribution (%)'))

plot_sector_attribution(attr)
plt.show()

## 8. XU100 Benchmark Comparison

In [ ]:
from advanced_analytics import plot_xu100_comparison, fetch_xu100

try:
    xu100 = fetch_xu100(
        start=str(strat_ret.index[0].date()),
        end=str(strat_ret.index[-1].date()),
    )
    plot_xu100_comparison(strat_ret, bench_ret, xu100)
    plt.show()
    
    # XU100 metrics
    xu100_aligned = xu100.reindex(strat_ret.index, method='ffill')
    xu100_ret = xu100_aligned.pct_change().dropna()
    common = strat_ret.index.intersection(xu100_ret.index)
    xu100_m = performance_metrics(xu100_ret.loc[common], name='XU100')
    full_report = pd.DataFrame([strat_metrics, bench_metrics, xu100_m])
    display(full_report)
except Exception as e:
    print(f"Could not download XU100 data: {e}")
    print("Skipping XU100 comparison.")

## 9. Transaction Cost Impact

In [ ]:
cost_levels = [0, 10, 30, 50, 100]  # in basis points
cost_results = []

for bps in cost_levels:
    res = run_backtest(prices, scores, cost_bps=bps)
    m = performance_metrics(res['strategy_returns'], name=f'{bps} bps')
    m['Total Turnover'] = round(res['total_turnover'], 2)
    m['Total Cost (bps)'] = round(res['total_cost_bps'], 1)
    cost_results.append(m)

cost_df = pd.DataFrame(cost_results)
print("Strategy performance under different transaction cost assumptions:\n")
display(cost_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar([str(b) for b in cost_levels], cost_df['Sharpe Ratio'], color='#4e79a7')
axes[0].set_title('Sharpe Ratio vs Transaction Cost', fontsize=13)
axes[0].set_xlabel('Cost (bps per rebalance)')
axes[0].set_ylabel('Sharpe Ratio')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar([str(b) for b in cost_levels], cost_df['Ann. Return (%)'], color='#f28e2b')
axes[1].set_title('Annualized Return vs Transaction Cost', fontsize=13)
axes[1].set_xlabel('Cost (bps per rebalance)')
axes[1].set_ylabel('Ann. Return (%)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 10. Sensitivity Analysis

⚠️ **Note:** The full sensitivity sweep runs ~60+ backtest iterations and may take a few minutes.

In [ ]:
from sensitivity import run_sensitivity_sweep, plot_sensitivity_heatmap, DEFAULT_GRIDS

print("Sweep: Risk Aversion × Tilt Strength")
df_ra_ts = run_sensitivity_sweep(
    prices, scores,
    'risk_aversion', DEFAULT_GRIDS['risk_aversion'],
    'tilt_strength', DEFAULT_GRIDS['tilt_strength'],
)

plot_sensitivity_heatmap(df_ra_ts, 'risk_aversion', 'tilt_strength', 'sharpe',
                         title='Sharpe: Risk Aversion × Tilt Strength')
plt.show()

plot_sensitivity_heatmap(df_ra_ts, 'risk_aversion', 'tilt_strength', 'excess_sharpe',
                         title='Excess Sharpe: Risk Aversion × Tilt Strength')
plt.show()

In [ ]:
print("Sweep: Max Weight × Lookback")
df_mw_lb = run_sensitivity_sweep(
    prices, scores,
    'max_weight', DEFAULT_GRIDS['max_weight'],
    'lookback', DEFAULT_GRIDS['lookback'],
)

plot_sensitivity_heatmap(df_mw_lb, 'max_weight', 'lookback', 'sharpe',
                         title='Sharpe: Max Weight × Lookback')
plt.show()

plot_sensitivity_heatmap(df_mw_lb, 'max_weight', 'lookback', 'excess_sharpe',
                         title='Excess Sharpe: Max Weight × Lookback')
plt.show()

---
**End of walkthrough.** All plots are also saved to the `reports/` folder when running the modules from the command line.